# OrionBelt Semantic Layer — Quickstart

This notebook demonstrates OrionBelt's semantic layer using the **TPC-H** benchmark dataset
in DuckDB. Everything runs locally — no cloud database needed.

**What you'll see:**

1. Create a TPC-H database and start the API
2. Explore the model: dimensions, measures, metrics, ER diagram
3. Compile OBML queries to SQL across 5 dialects
4. Execute queries and see real results
5. Filters: WHERE, HAVING, and filter groups (OR logic)
6. Multi-fact CFL (Composite Fact Layer) in action
7. Metrics (cross-measure formulas)
8. Dimension exclusion (anti-join)
9. Cumulative metrics (running totals & rolling windows)
10. Period-over-period metrics (YoY growth & MoM change)
11. Lineage and search

## Step 0: Create the TPC-H Database

DuckDB's built-in TPC-H generator creates 8 tables with realistic supply-chain data.

In [ ]:
from notebook_setup import (
    create_tpch_database, start_api, api,
    show_result, show_sql, show_json, show_mermaid,
)

create_tpch_database()

## Step 1: Start the API

In [ ]:
SID, MID = start_api()

## Step 2: Explore the Model

The TPC-H model defines 12 dimensions, 10 measures, and 8 metrics
(including derived, cumulative, and period-over-period).
All shortcut endpoints auto-resolve to the loaded model.

In [ ]:
# Dimensions — the business concepts you can group by
dims = api("GET", "/v1/dimensions")
print("Dimensions:")
for d in dims:
    print(f"  {d['name']:20s}  ({d['data_object']}.{d['column']})")

In [ ]:
# Measures — aggregations over fact table columns
print("Measures:")
for m in api("GET", "/v1/measures"):
    agg = m.get("aggregation", "")
    expr = m.get("expression", "")
    detail = expr if expr else agg
    print(f"  {m['name']:20s}  {detail}")

In [ ]:
# Metrics — cross-measure formulas
print("Metrics:")
for m in api("GET", "/v1/metrics"):
    print(f"  {m['name']:20s}  = {m['expression']}")

### ER Diagram

The model's data objects, columns, and join relationships at a glance.

In [ ]:
er = api("GET", f"/v1/sessions/{SID}/models/{MID}/diagram/er?theme=dark")
show_mermaid(er["mermaid"])

## Step 3: Compile Queries

Write queries using business names — OrionBelt compiles them to optimized,
dialect-specific SQL with automatic joins.

### Star schema: Revenue by Region

In [ ]:
query = """
select:
  dimensions: [Region]
  measures: [Revenue, Order Count]
"""
result = api("POST", "/v1/query/sql?dialect=duckdb", query)
show_result(result, query)

The compile result also includes an **explain plan** — which planner was chosen,
the base object, the join path with reasons, and filter counts.

In [ ]:
import yaml
from notebook_setup import show_yaml
from IPython.display import display, HTML

display(HTML(show_yaml(yaml.dump(result["explain"], default_flow_style=False, sort_keys=False))))

## Step 4: Execute Queries (Real Data!)

The `/v1/query/execute` endpoint compiles OBML and runs the SQL against the database,
returning actual results.

In [ ]:
query = """
select:
  dimensions: [Region]
  measures: [Revenue, Order Count, Total Quantity]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
query = """
select:
  dimensions: [Nation, Market Segment]
  measures: [Revenue]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Filters: WHERE, HAVING, and Filter Groups

In [ ]:
query = """
select:
  dimensions: [Ship Mode]
  measures: [Revenue, Order Count]
where:
  - field: Order Priority
    op: "="
    value: 1-URGENT
having:
  - field: Order Count
    op: ">"
    value: 100
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Filter Groups: OR Logic

Use filter groups to combine conditions with OR. Here we find revenue for
orders that are either **urgent** or **high priority**.

In [ ]:
query = """
select:
  dimensions: [Ship Mode]
  measures: [Revenue, Order Count]
where:
  - logic: or
    filters:
      - field: Order Priority
        op: "="
        value: 1-URGENT
      - field: Order Priority
        op: "="
        value: 2-HIGH
order_by:
  - field: Revenue
    direction: desc
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 5: Multi-Fact CFL (Composite Fact Layer)

When a query references measures from **independent fact tables**, OrionBelt
automatically uses the CFL planner. It generates `UNION ALL BY NAME` (DuckDB,
Snowflake) or `UNION ALL` with NULL padding (other dialects) to prevent
fan-trap row multiplication.

Here, **Revenue** comes from `Line Items` and **Available Stock** from `Part Supply` —
two independent facts that share `Parts` as a conformed dimension.

In [ ]:
query = """
select:
  dimensions: [Brand]
  measures: [Revenue, Available Stock, Inventory Value]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Secondary Join Path: `usePathNames`

The CFL above treats **Revenue** (Line Items) and **Available Stock** (Part Supply) as
independent facts because there is no primary join between them.

But TPC-H has a composite foreign key: `lineitem.(l_partkey, l_suppkey)` →
`partsupp.(ps_partkey, ps_suppkey)`. This is modelled as a **secondary join** with
`pathName: lineitem_partsupp`.

Activating it with `usePathNames` makes Part Supply reachable from Line Items,
so both tables resolve through a **single star join** — no CFL needed. This enables
the cross-object measure **Supply Cost** (`SUM(l_quantity * ps_supplycost)`) and
the **Gross Margin** metric.

In [ ]:
query = """
select:
  dimensions: [Brand, Supplier]
  measures: [Revenue, Supply Cost, Gross Margin]
usePathNames:
  - source: Line Items
    target: Part Supply
    pathName: lineitem_partsupp
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 6: Metrics (Cross-Measure Formulas)

Metrics are calculated from measures: `{[Total Discount]} / {[Gross Amount]}`.
OrionBelt resolves the formula and generates the correct SQL expression.

In [ ]:
query = """
select:
  dimensions: [Market Segment]
  measures: [Revenue, Gross Amount, Discount Rate, Average Order Value]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 7: Dimension Exclusion (Anti-Join)

`dimensionsExclude` inverts a dimension-only query: instead of existing combinations,
it returns combinations that **do not exist**. OrionBelt generates a `CROSS JOIN`
of all distinct dimension values, then subtracts existing pairs via `EXCEPT`.

*Which customers have never bought which brands?*

In [ ]:
# Which customers have never bought which brands?
query = """
select:
  dimensions: [Customer, Brand]
dimensionsExclude: true
order_by:
  - field: Customer
  - field: Brand
limit: 20
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 8: Cumulative Metrics (Running Totals & Rolling Windows)

Cumulative metrics compute **running totals** or **rolling window** aggregations
over a time dimension. OrionBelt wraps the planner output in a CTE with
`SUM(...) OVER (ORDER BY date ROWS ...)` window functions.

- **Running Revenue** — unbounded running total from the first date onward
- **Rolling 3m Revenue** — sum over the current row and the 2 preceding periods

In [ ]:
# Running total of revenue by month
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Running Revenue]
order_by:
  - field: Order Date
limit: 12
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
# Rolling 3-month window alongside running total
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Running Revenue, Rolling 3m Revenue]
order_by:
  - field: Order Date
limit: 12
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 9: Period-over-Period Metrics (YoY Growth & MoM Change)

Period-over-period (PoP) metrics compare each time period against a previous one.
OrionBelt generates a **4-CTE date spine** architecture:

1. `date_range` — auto-discovers MIN/MAX date from the data (with all WHERE filters pushed down)
2. `date_spine` — generates a date series at the configured grain with a previous-period lookup
3. `pop_base` — aggregates measures using the spine as the driver (LEFT JOIN to facts)
4. `pop_compare` — self-joins to compute the comparison (percent change, difference, ratio, or previous value)

No hardcoded date ranges needed — the spine is auto-discovered from the data.

In [ ]:
# Year-over-year revenue growth by month
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Revenue YoY Growth]
order_by:
  - field: Order Date
limit: 24
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
# Month-over-month revenue change (absolute difference) by region
query = """
select:
  dimensions:
    - "Order Date:month"
    - Region
  measures: [Revenue, Revenue MoM Change]
order_by:
  - field: Order Date
  - field: Region
limit: 20
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 10: Lineage & Search

In [ ]:
from notebook_setup import show_json
from IPython.display import display, HTML

explain = api("GET", "/v1/explain/Revenue")
display(HTML(show_json(explain)))

In [ ]:
# Search for anything related to "order"
results = api("POST", "/v1/find", {"query": "order"})
for item in results["results"]:
    print(f"  [{item['type']:10s}]  {item['name']}")

---

## Next Steps

- **Gradio UI** — Visual model explorer with ER diagrams, SQL preview, and query builder:
  ```bash
  docker pull ralforion/orionbelt-ui
  docker run -p 7860:7860 -e API_BASE_URL=http://host.docker.internal:8080 ralforion/orionbelt-ui
  ```
- **Arrow Flight SQL** — Connect DBeaver, Tableau, or Power BI directly — see [docs/drivers.md](../docs/drivers.md)
- **DB-API 2.0 Drivers** — Use `ob-duckdb`, `ob-postgres`, `ob-snowflake`, etc. from Python
- **MCP Server** — Connect AI assistants via [orionbelt-semantic-layer-mcp](https://github.com/ralfbecher/orionbelt-semantic-layer-mcp)
- **Full documentation** — [ralforion.com/orionbelt-semantic-layer](https://ralforion.com/orionbelt-semantic-layer/)